In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks/practice")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


    # 01 · Agent identity and principals — practice

    **Primer sections:** §3.1–§3.3. Reproduce the worked notebook from memory: build an agent
    principal, match it against IAM-style members, issue a runtime certificate, mint a bound token,
    prove replay fails, and label the authority mode.

> **How to use this practice notebook.** Every `____` is a blank you must fill (a name, an
> argument, an expression); a `raise NotImplementedError("fill me")` means "write the body".
> Each exercise ends with `assert` checks — run the cell, and if it is silent you got it right.
> The completed version lives in `notebooks/solutions/`.

In [ ]:
from agentsec.logging_utils import quiet_logs

quiet_logs()

import datetime as dt
import json

import jwt  # PyJWT, only used to *peek* at unverified claims for display

from agentsec.identity import (
    AgentIdentity,
    AuthorityContext,
    AuthorityMode,
    BindingMismatch,
    LocalRuntimeCA,
    PrincipalSet,
    TokenIssuer,
    UserPrincipal,
    member_matches,
)


def peek(token: str) -> dict:
    """Display helper: decode WITHOUT verifying. Never do this for authorization decisions."""
    return jwt.decode(token, options={"verify_signature": False})

## Exercise 1 — name the principal

Build the Agent Engine identity for project number `987654321098`, location `us-central1`,
engine `support-agent`, in organisation `123456789012`, and derive its IAM member string.

In [ ]:
agent = AgentIdentity.for_agent_engine(
    project_number="987654321098", location="us-central1", engine_id="support-agent", org_id=____
)
iam_member = ____

print(agent.spiffe_id)
print(iam_member)
assert agent.trust_domain == "agents.global.org-123456789012.system.id.goog"
assert iam_member.startswith("principal://") and AgentIdentity.parse(iam_member) == agent
assert agent.platform_container == "aiplatform/projects/987654321098"

## Exercise 2 — principalSet matching is exact

Write the `principalSet://` member that selects **every agent in project 987654321098**, and the
one that selects **every Agent Engine agent in the organisation**. Then implement `selects()` that
returns whether an arbitrary IAM member string selects the agent (hint: the library has a one-liner).

In [ ]:
TD = "agents.global.org-123456789012.system.id.goog"
other_project_agent = AgentIdentity.for_agent_engine(
    project_number="111111111111", location="us-central1", engine_id="marketing-agent", org_id="123456789012"
)

project_set_member = f"principalSet://{TD}/attribute.____/aiplatform/projects/987654321098"
platform_set_member = f"principalSet://{TD}/attribute.____/aiplatform"

def selects(member: str, who: AgentIdentity) -> bool:
    raise NotImplementedError("fill me")

assert PrincipalSet.parse(project_set_member).matches(agent)
assert not PrincipalSet.parse(project_set_member).matches(other_project_agent)
assert PrincipalSet.parse(platform_set_member).matches(agent)
assert PrincipalSet.parse(platform_set_member).matches(other_project_agent)
# a prefix of the project number must not match
assert not selects(f"principalSet://{TD}/attribute.platformContainer/aiplatform/projects/98765432109", agent)
assert selects(agent.iam_principal, agent) and not selects("user:ana@customer.example", agent)
print("principalSet matching: exact segments, as IAM does")

## Exercise 3 — the runtime issues a certificate with the SPIFFE ID in the SAN

Issue a 24-hour certificate for the agent from the local runtime CA and check the SAN.

In [ ]:
ca = LocalRuntimeCA()
cert = ca.issue(____, ttl=____)

assert cert.spiffe_id == agent.spiffe_id
assert ca.verify(cert.certificate) and cert.is_valid_at()
assert cert.not_after - dt.datetime.now(dt.UTC) < dt.timedelta(hours=25)
assert not LocalRuntimeCA("some-other-runtime").verify(cert.certificate)
print("thumbprint (x5t#S256):", cert.thumbprint)

## Exercise 4 — bind the token, then try to replay it

Mint the agent's own-authority token for audience `https://api.acme.example`, verify it with the
right certificate, and show that both a *different* certificate and a bare bearer presentation
are rejected with `BindingMismatch`.

In [ ]:
issuer = TokenIssuer()
AUD = "https://api.acme.example"
token = issuer.____(cert, audience=AUD, scope="orders:read")
stolen_runtime_cert = ca.issue(agent)

claims = issuer.verify(token, audience=AUD, presented_thumbprint=____)
assert claims.cnf["x5t#S256"] == cert.thumbprint and claims.subject == agent.spiffe_id

def replay_rejected(**kw) -> bool:
    try:
        issuer.verify(token, audience=AUD, **kw)
        return False
    except ____:
        return True

assert replay_rejected(presented_thumbprint=stolen_runtime_cert.thumbprint)
assert replay_rejected()  # plain bearer use
print("bound token: accepted with the right cert, rejected from another cert and as bare bearer")

## Exercise 5 — label the authority

Create a delegated context for Ana with scopes `{"orders:read"}`, and an own-authority context
with no user. `audit_identities()` must show the user only in the delegated case. Then add a hop
to a sub-agent and confirm the scopes did not widen.

In [ ]:
ana = UserPrincipal(subject="u-ana", email="ana@customer.example", tenant="acme")
sub_agent = AgentIdentity.for_agent_engine(
    project_number="987654321098", location="us-central1", engine_id="refunds-subagent", org_id="123456789012"
)

delegated = AuthorityContext.____(agent, ana, scopes={"orders:read"})
own = AuthorityContext.____(agent)
hop = delegated.____(sub_agent)

assert delegated.audit_identities() == {"agent": agent.spiffe_id, "user": "ana@customer.example", "authority": "delegated"}
assert own.audit_identities() == {"agent": agent.spiffe_id, "user": None, "authority": "own"}
assert hop.chain == (sub_agent.spiffe_id, agent.spiffe_id) and hop.scopes == frozenset({"orders:read"})
assert AuthorityContext.from_claims(claims).mode is AuthorityMode.OWN
print(json.dumps(delegated.audit_identities(), indent=2))
print(json.dumps(hop.audit_identities(), indent=2))

**In one sentence:** "An agent is its own principal — a SPIFFE ID attested by a runtime
certificate, tokens bound to that certificate, fleet policy via principalSets that match exact
segments, and an explicit own-vs-delegated authority on every action."